<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/NLPMusteriYorumlari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Veri Yükleme ve İlk Keşif

In [ ]:
import pandas as pd
df = pd.read_csv('/content/trendyol.csv', sep=';')
print("DataFrame'in ilk 5 satırı:")
display(df.head())

print("\nDataFrame Bilgisi:")
df.info()

In [ ]:
print("DataFrame'in Boyutları (satır, sütun):")
print(df.shape)

### 2. Veri Ön İşleme ve Temizleme

In [ ]:
new_cols = df.columns[0].split(',')
df[new_cols] = df[df.columns[0]].str.rsplit(',', n=2, expand=True)
df = df.drop(columns=[df.columns[0], 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3'])
df['review_rating'] = pd.to_numeric(df['review_rating'], errors='coerce')

print("\nEksik değer kontrolü (temizleme öncesi):\n", df.isnull().sum())
df.dropna(subset=['review_body'], inplace=True)

print("\nTemizlenmiş DataFrame'in ilk 5 satırı:")
display(df.head())
print("\nTemizlenmiş DataFrame Bilgisi:")
df.info()

### 3. Metin Ön İşleme ve Normalizasyon

In [ ]:
%pip install nltk

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

turkish_stopwords = stopwords.words('turkish')

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = text.translate(str.maketrans('', '', string.punctuation))

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in turkish_stopwords]
    return " ".join(tokens)

df['processed_review_body'] = df['review_body'].apply(preprocess_text)

print("İşlenmiş yorum gövdesinin ilk 5 örneği:")
for i, row in df.head(5).iterrows():
    print(f"Orijinal: {row['review_body']}")
    print(f"İşlenmiş: {row['processed_review_body']}\n")

### 4. Morfolojik Analiz

In [ ]:
%pip install zeyrek

In [ ]:
from zeyrek import MorphAnalyzer

morph = MorphAnalyzer()

sample_word = "dökülmelerimi"
analysis = morph.analyze(sample_word)

print(f"'{sample_word}' kelimesinin morfolojik analizi:")
for analysis_result_list in analysis:
    for an_parse_object in analysis_result_list:
        print(f"  Kök: {an_parse_object.lemma}, Lemma: {an_parse_object.lemma}, Tür: {an_parse_object.pos}, Ekler (morphemes): {str(an_parse_object.morphemes)}")

sample_processed_review = df['processed_review_body'].iloc[0]
print(f"\nÖrnek işlenmiş yorum: '{sample_processed_review}'")

words_to_analyze = sample_processed_review.split()[:5]
print(f"\nİlk beş kelimenin morfolojik analizi:")
for word in words_to_analyze:
    analysis_word = morph.analyze(word)
    print(f"  '{word}':")
    if analysis_word:
        for analysis_result_list in analysis_word:
            for an_parse_object in analysis_result_list:
                print(f"    Kök: {an_parse_object.lemma}, Lemma: {an_parse_object.lemma}, Tür: {an_parse_object.pos}, Ekler (morphemes): {str(an_parse_object.morphemes)}")
    else:
        print("    Analiz edilemedi.")

In [ ]:
from joblib import Parallel, delayed
import multiprocessing
from zeyrek import MorphAnalyzer

_morph_analyzer_instance = None

def get_morph_analyzer():
    global _morph_analyzer_instance
    if _morph_analyzer_instance is None:
        _morph_analyzer_instance = MorphAnalyzer()
    return _morph_analyzer_instance

def lemmatize_text_worker(text):
    morph = get_morph_analyzer()

    if not isinstance(text, str):
        return ""
    words = text.split()
    lemmas = []
    for word in words:
        analysis = morph.analyze(word)
        if analysis:
            if analysis[0]:
                lemmas.append(analysis[0][0].lemma)
            else:
                lemmas.append(word)
        else:
            lemmas.append(word)
    return " ".join(lemmas)

num_cores = multiprocessing.cpu_count()
print(f"Using {num_cores} CPU cores for parallel lemmatization.")


lemmatized_reviews_list = Parallel(n_jobs=num_cores, verbose=10)(
    delayed(lemmatize_text_worker)(text) for text in df['processed_review_body'].tolist()
)

df['lemmatized_review_body'] = lemmatized_reviews_list

print("Lemmatize edilmiş yorum gövdesinin ilk 5 örneği:")
for i, row in df.head(5).iterrows():
    print(f"Orijinal: {row['review_body']}")
    print(f"İşlenmiş: {row['processed_review_body']}")
    print(f"Lemmatize Edilmiş: {row['lemmatized_review_body']}\n")

print("DataFrame'in ilk 5 satırı (yeni sütun ile birlikte):")
display(df.head())

### 5. BERT Tabanlı NER Modeli


In [ ]:
pip install stanza

In [ ]:
import stanza

stanza.download('tr')

print("Stanza Türkçe modeli başarıyla indirildi.")

### 6. Stanza NER Pipeline

In [ ]:
import stanza

nlp = stanza.Pipeline(lang='tr', processors='tokenize,ner')

sample_reviews = df['lemmatized_review_body'].head(5).tolist()

print("Örnek Lemmatize Edilmiş Yorumlar Üzerinde NER Uygulaması:")
for i, text in enumerate(sample_reviews):
    if text.strip() == "":
        continue
    doc = nlp(text)
    print(f"\n--- Yorum {i+1} ---")
    print(f"Metin: {text}")
    print("Tanınan Varlıklar:")
    if doc.entities:
        for ent in doc.entities:
            print(f"  Varlık: {ent.text}, Tür: {ent.type}")
    else:
        print("  Varlık bulunamadı.")

### 7. Bağımlılık Analizi


In [ ]:
import stanza

nlp_dep = stanza.Pipeline(lang='tr', processors='tokenize,pos,lemma,depparse', use_gpu=False)

print("Stanza Türkçe Bağımlılık Analizi pipeline'ı başarıyla başlatıldı.")

sample_review_for_dep = df['processed_review_body'].iloc[0]

print(f"\nBağımlılık Analizi Uygulanacak Örnek Metin: '{sample_review_for_dep}'")

doc_dep = nlp_dep(sample_review_for_dep)

print("\nKelime Bağımlılık Analizi Sonuçları:")
for sent in doc_dep.sentences:
    for word in sent.words:

        head_text = sent.words[word.head-1].text if word.head > 0 else "ROOT"
        print(f"ID: {word.id}\tKelime: {word.text}\tLemma: {word.lemma}\tPOS: {word.pos}\tHead Kelime: {head_text}\tDeprel: {word.deprel}")

### Bağımlılık Analizi Görselleştirme (Ağ Grafiği)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Grafik oluşturma fonksiyonu
def visualize_dependency_parse(doc_dep):
    G = nx.DiGraph()

    # Kök kelimeyi ekle
    G.add_node(0, label='ROOT')

    # Kelimeleri düğüm olarak ekle
    for sent in doc_dep.sentences:
        for word in sent.words:
            G.add_node(word.id, label=word.text)

    # Bağımlılık ilişkilerini kenar olarak ekle
    edge_labels = {}
    for sent in doc_dep.sentences:
        for word in sent.words:
            head_id = word.head
            if head_id == 0:
                G.add_edge(head_id, word.id)
                edge_labels[(head_id, word.id)] = word.deprel
            else:
                G.add_edge(head_id, word.id)
                edge_labels[(head_id, word.id)] = word.deprel

    plt.figure(figsize=(15, 10))
    pos = nx.spring_layout(G, k=0.7, iterations=50)

    # Düğümleri çiz
    nx.draw_networkx_nodes(G, pos, node_size=3000, node_color="skyblue", alpha=0.9)

    # Kenarları çiz
    nx.draw_networkx_edges(G, pos, edgelist=G.edges(), arrowstyle="<-", arrowsize=20, edge_color="gray", width=1)

    # Düğüm etiketlerini çiz (kelimeler)
    node_labels = {node: G.nodes[node]['label'] for node in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=10, font_weight="bold")

    # Kenar etiketlerini çiz (bağımlılık türleri)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=9)

    plt.title(f"'{sample_review_for_dep}' için Bağımlılık Ağacı", size=15)
    plt.axis('off')
    plt.show()

visualize_dependency_parse(doc_dep)